In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

matches.isnull().sum()
deliveries.isnull().sum()

match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64

Data cleaning


In [2]:
team_replacements = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}

for col in ['team1', 'team2', 'winner', 'toss_winner']:
    matches[col] = matches[col].replace(team_replacements)

for col in ['batting_team', 'bowling_team']:
    deliveries[col] = deliveries[col].replace(team_replacements)    

In [3]:
matches['team1'].unique()           

<StringArray>
['Royal Challengers Bangalore',                'Punjab Kings',
              'Delhi Capitals',              'Mumbai Indians',
       'Kolkata Knight Riders',            'Rajasthan Royals',
         'Sunrisers Hyderabad',         'Chennai Super Kings',
        'Kochi Tuskers Kerala',               'Pune Warriors',
               'Gujarat Lions',      'Rising Pune Supergiant',
        'Lucknow Super Giants',              'Gujarat Titans',
 'Royal Challengers Bengaluru']
Length: 15, dtype: str

In [4]:
deliveries['batting_team'].unique()

<StringArray>
[      'Kolkata Knight Riders', 'Royal Challengers Bangalore',
         'Chennai Super Kings',                'Punjab Kings',
            'Rajasthan Royals',              'Delhi Capitals',
              'Mumbai Indians',         'Sunrisers Hyderabad',
        'Kochi Tuskers Kerala',               'Pune Warriors',
      'Rising Pune Supergiant',               'Gujarat Lions',
        'Lucknow Super Giants',              'Gujarat Titans',
 'Royal Challengers Bengaluru']
Length: 15, dtype: str

In [5]:
deliveries['is_boundary'] = deliveries['batsman_runs'].apply(
    lambda x: 1 if x in [4, 6] else 0
)

In [6]:
deliveries['is_dot'] = deliveries['total_runs'].apply(
    lambda x: 1 if x == 0 else 0
)

In [7]:
deliveries['is_wicket'] = deliveries['dismissal_kind'].notnull().astype(int)

In [8]:
#Over Phase Column
def over_phase(over):
    if over <= 6:
        return 'Powerplay'
    elif over <= 15:
        return 'Middle Overs'
    else:
        return 'Death Overs'

deliveries['phase'] = deliveries['over'].apply(over_phase)

In [9]:
deliveries.to_csv(
    "deliveries_cleaned.csv",
    index=False
)

matches.to_csv(
    "matches_cleaned.csv",
    index=False
)

ORGANE CAP TABLE


In [10]:
orange_cap = deliveries.groupby('batter').agg({
    'batsman_runs':'sum',
    'ball':'count',
    'is_boundary':'sum'
}).reset_index()

#Rename columns
orange_cap.columns = [
    'batter',
    'runs',
    'balls',
    'boundaries'
]

#strike rate
orange_cap['strike_rate'] = (
    orange_cap['runs']
    /
    orange_cap['balls']
) * 100

#sort players
orange_cap = orange_cap.sort_values(
    by='runs',
    ascending=False
)

orange_cap.head(10)

#save table
orange_cap.to_csv(
    "orange_cap.csv",
    index=False
)

BOWLING TABLE

In [11]:
purple_cap = deliveries.groupby('bowler').agg({
    'is_wicket':'sum',
    'ball':'count',
    'total_runs':'sum',
    'is_dot':'sum'
}).reset_index()

purple_cap.columns = [
    'bowler',
    'wickets',
    'balls',
    'runs_given',
    'dot_balls'
]

#ECONOMY RATE
purple_cap['overs'] = (
    purple_cap['balls'] / 6
)

purple_cap['economy'] = (
    purple_cap['runs_given']
    /
    purple_cap['overs']
)

#sort bowlers
purple_cap = purple_cap.sort_values(
    by='wickets',
    ascending=False
)

purple_cap.head(10)

#save table
purple_cap.to_csv(
    "purple_cap.csv",
    index=False
)

TEAM ANALYTICS


In [14]:
#Team wins table
team_wins = matches['winner'].value_counts().reset_index()

team_wins.columns = [
    'team',
    'wins'
]

team_wins.to_csv(
    "team_wins.csv",
    index=False
)


#Toss impact analysis
matches['toss_match_win'] = np.where(
    matches['toss_winner'] == matches['winner'],
    'Yes',
    'No'
)

#Toss impact %
matches['toss_match_win'].value_counts(normalize=True) * 100

#Updated matches
matches.to_csv(
    "matches_cleaned.csv",
    index=False
)

VENUE ANALYTICS

In [18]:
#First innings data
first_innings = deliveries[
    deliveries['inning'] == 1
]

#Match wise first innings Score
first_innings_score = first_innings.groupby(
    'match_id'
)['total_runs'].sum().reset_index()

first_innings_score.columns = [
    'match_id',
    'first_innings_score'
]

#Merge with matches table
venue_analysis = matches.merge(
    first_innings_score,
    left_on='id',
    right_on='match_id'
)

# Average score per venue
venue_avg_score = venue_analysis.groupby(
    'venue'
)['first_innings_score'].mean().reset_index()


# RENAME COLUMN
venue_avg_score.columns = [
    'venue',
    'avg_first_innings_score'
]

#Sorting highest Scoring grounds
venue_avg_score = venue_avg_score.sort_values(
    by='avg_first_innings_score',
    ascending=False
)


#check results
venue_avg_score.head(10)

# save venue
venue_avg_score.to_csv(
    "venue_avg_score.csv",
    index=False
)

CHASING ANALYSIS

In [21]:
#Create toss decision insights
matches['toss_decision'].value_counts()

#Venue wise toss decisions
venue_toss = matches.groupby(
    ['venue', 'toss_decision']
).size().reset_index(name='count')

venue_toss.to_csv(
    "venue_toss.csv",
    index=False
)

Team Consistency Analysis

In [24]:
team1 = matches['team1'].value_counts()
team2 = matches['team2'].value_counts()


#Combine
total_matches = (
    team1 + team2
).reset_index()

total_matches.columns = [
    'team',
    'matches_played'
]

#Margin wins
team_consistency = total_matches.merge(
    team_wins,
    on='team'
)

#Win pecentage
team_consistency['win_percentage'] = (
    team_consistency['wins']
    /
    team_consistency['matches_played']
) * 100

#Sort
team_consistency = team_consistency.sort_values(
    by='win_percentage',
    ascending=False
)

#Save
team_consistency.to_csv(
    "team_consistency.csv",
    index=False
)